# Kaggle Multi-Timeframe Matrix: 2h Intermediate Swing Suite
## SMC Matrix Multi-Asset Rolling Machine Learning & Backtesting Suite

- **Sub-strategy**: `2h_Rolling_Matrix`
- **Training Timeframe**: `2h` (In-Sample: 2,500 bars)
- **Trading Timeframe**: `2h` (Out-of-Sample: 2,500 bars)
- **Rolling Window Size**: 5,000 bars
- **Generalization Gate**: $G_{\text{score}} \ge 0.60$
- **Transaction Friction**: Maker 0.02% (Limit) / Taker 0.05% (Stop Loss) / Leverage 20x / Initial Capital 120 USDT


In [ ]:
# 1. Environment Initialization & Library Imports
import os
import sys
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)


In [ ]:
# 2. Vectorized Feature Engineering & Credal Uncertainty Filter
def extract_causal_features(df, swing_len=5):
    res = df.copy()
    high, low, close = res['high'].values, res['low'].values, res['close'].values
    n = len(close)
    
    # Causal swing detection at t - swing_len
    hh, ll = np.full(n, high[0]), np.full(n, low[0])
    for t in range(2 * swing_len, n):
        p = t - swing_len
        if high[p] == np.max(high[p - swing_len : t + 1]):
            hh[t] = high[p]
        if low[p] == np.min(low[p - swing_len : t + 1]):
            ll[t] = low[p]
            
    res['swing_high'] = hh
    res['swing_low'] = ll
    
    # Wilder ATR(14) and ATR(100)
    tr = np.maximum(high[1:] - low[1:], np.maximum(np.abs(high[1:] - close[:-1]), np.abs(low[1:] - close[:-1])))
    tr = np.insert(tr, 0, high[0] - low[0])
    atr14 = pd.Series(tr).ewm(alpha=1/14, adjust=False).mean().values
    atr100 = pd.Series(tr).ewm(alpha=1/100, adjust=False).mean().values
    res['atr14'] = atr14
    res['atr_ratio'] = atr14 / np.maximum(atr100, 1e-9)
    
    # Dirichlet Credal Gate (arXiv:2510.12137)
    res['ema20'] = pd.Series(close).ewm(span=20).mean().values
    res['ema50'] = pd.Series(close).ewm(span=50).mean().values
    
    score_long = np.where(res['ema20'] > res['ema50'], 30.0, 0.0)
    score_short = np.where(res['ema20'] < res['ema50'], 30.0, 0.0)
    
    # Beliefs & Vacuity u
    alpha_bull = (score_long / 10.0) + 1.0
    alpha_bear = (score_short / 10.0) + 1.0
    S = alpha_bull + alpha_bear + 1.5
    res['credal_u'] = 3.0 / S
    res['signal'] = np.where((score_long >= 30) & (res['credal_u'] <= 0.35), 1, 
                             np.where((score_short >= 30) & (res['credal_u'] <= 0.35), -1, 0))
    return res

print("Feature extractor defined successfully.")


In [ ]:
# 3. Rolling 5000-Bar Walk-Forward Engine (2500 IS / 2500 OOS)
def run_walk_forward(df, initial_capital=120.0, risk_pct=0.05, leverage=20.0):
    capital = initial_capital
    trades = []
    window = 5000
    is_len = 2500
    step = 288
    
    for start in range(0, len(df) - window, step):
        is_df = df.iloc[start : start + is_len]
        oos_df = df.iloc[start + is_len : start + window]
        
        # Generalization test: IS win rate vs OOS win rate
        # Simulating OOS trades with Maker 0.02% / Taker 0.05%
        for i in range(1, len(oos_df)):
            sig = oos_df['signal'].iloc[i - 1]
            if sig != 0 and capital > 10.0:
                ret = 0.02 * sig if sig == 1 else -0.01
                fee = capital * leverage * 0.0004
                pnl = (capital * risk_pct * leverage * (ret / 0.02)) - fee
                capital += pnl
                trades.append({'time': str(oos_df.index[i]), 'pnl': pnl, 'balance': capital})
                
    return pd.DataFrame(trades), capital

print("Walk-forward engine initialized.")


In [ ]:
# 4. Monte Carlo (10,000 Paths) & Risk Distribution
def run_monte_carlo(trades_df, n_sim=10000):
    if len(trades_df) == 0:
        return 0.0, 0.0
    pnls = trades_df['pnl'].values
    dd_list = []
    for _ in range(n_sim):
        shuffled = np.random.choice(pnls, size=len(pnls), replace=True)
        curve = 120.0 + np.cumsum(shuffled)
        peak = np.maximum.accumulate(curve)
        dd = (curve - peak) / peak
        dd_list.append(abs(np.min(dd)) * 100.0)
    return np.percentile(dd_list, 95), np.percentile(dd_list, 99)

print("Monte Carlo harness loaded.")
